In [ ]:
# ============================================================================
# Dataset Truncation Diagnostic Script (Nepali Prompt, 1024 tokens)
# ============================================================================

import json, os
import sentencepiece as spm

# ----------------------------
# CONFIG
# ----------------------------
TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
JSONL_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Normalized\train_plus_val_norm.jsonl"

TOTAL_SEQ_LENGTH = 1024
MAX_SOURCE_LENGTH = int(0.75 * TOTAL_SEQ_LENGTH)
MAX_TARGET_LENGTH = int(0.25 * TOTAL_SEQ_LENGTH)

PROMPT_TEMPLATE = "यो लेखको संक्षेप गर्नुहोस्:\n{text}\nसारांश:\n"
PROMPT_TOKEN_BUFFER = 10  # fixed subtraction as requested

# ----------------------------
# LOAD TOKENIZER
# ----------------------------
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab: {sp.vocab_size()})\n")

# ----------------------------
# PROMPT TOKEN ANALYSIS
# ----------------------------
dummy = "बिनित"
prompt_tokens = sp.encode(PROMPT_TEMPLATE.format(text=dummy))
prompt_overhead = len(prompt_tokens) - len(sp.encode(dummy))

effective_article_budget = MAX_SOURCE_LENGTH - PROMPT_TOKEN_BUFFER

print("=" * 80)
print("PROMPT TOKEN ANALYSIS")
print("=" * 80)
print(f"Total seq length:        {TOTAL_SEQ_LENGTH}")
print(f"Source budget (75%):     {MAX_SOURCE_LENGTH}")
print(f"Target budget (25%):     {MAX_TARGET_LENGTH}")
print(f"Prompt overhead:         {prompt_overhead}")
print(f"Article tokens allowed:  {effective_article_budget}")
print("=" * 80 + "\n")

# ----------------------------
# LOAD DATA
# ----------------------------
with open(JSONL_PATH, encoding="utf-8") as f:
    data = [
        json.loads(line) for line in f
        if line.strip() and "text" in line and "summary" in line
    ]

print(f"✓ Loaded {len(data)} samples from {os.path.basename(JSONL_PATH)}\n")

# ----------------------------
# VIOLATION ANALYSIS
# ----------------------------
violations = {
    "article": [],
    "summary": [],
    "combined": []
}

for i, item in enumerate(data):
    text_len = len(sp.encode(item["text"]))
    summary_len = len(sp.encode(item["summary"]))
    combined_len = len(sp.encode(PROMPT_TEMPLATE.format(text=item["text"]) + item["summary"]))

    # if text_len > effective_article_budget:
    #     violations["article"].append((i, text_len))
    if text_len > MAX_SOURCE_LENGTH:
        violations["article"].append((i, text_len))
    if summary_len > MAX_TARGET_LENGTH:
        violations["summary"].append((i, summary_len))
    if combined_len > TOTAL_SEQ_LENGTH:
        violations["combined"].append((i, combined_len))

# ----------------------------
# REPORT VIOLATIONS
# ----------------------------
print("=" * 80)
print("TOKEN LIMIT VIOLATIONS")
print("=" * 80)

for k, label in [
    # ("article", f"Articles > {effective_article_budget} tokens"),
    ("article", f"Articles > {MAX_SOURCE_LENGTH} tokens"),
    ("summary", f"Summaries > {MAX_TARGET_LENGTH} tokens"),
    ("combined", f"Samples > {TOTAL_SEQ_LENGTH} tokens"),
]:
    print(f"\n{label}: {len(violations[k])}")
    for idx, length in violations[k][:10]:
        print(f"  Sample {idx}: {length} tokens")

print("\n(Note: showing first 10 per category)")
print("=" * 80 + "\n")

# ----------------------------
# SAMPLE PREVIEW (FIRST 3)
# ----------------------------
for i, item in enumerate(data[:3]):
    print("=" * 80)
    print(f"SAMPLE {i+1}")

    text_tokens = sp.encode(item["text"])
    summary_tokens = sp.encode(item["summary"])
    full_tokens = sp.encode(PROMPT_TEMPLATE.format(text=item["text"]) + item["summary"])

    print(f"Text tokens:     {len(text_tokens)}")
    print(f"Summary tokens:  {len(summary_tokens)}")
    print(f"Combined tokens: {len(full_tokens)}")

    if len(full_tokens) > TOTAL_SEQ_LENGTH:
        truncated = sp.decode(full_tokens[:TOTAL_SEQ_LENGTH])
        print(f"Truncated to {TOTAL_SEQ_LENGTH} tokens")
        print(truncated[:1024] + "...\n")
    else:
        print("No truncation needed\n")


✓ Tokenizer loaded (vocab: 16384)

PROMPT TOKEN ANALYSIS
Total seq length:        1024
Source budget (75%):     768
Target budget (25%):     256
Prompt overhead:         10
Article tokens allowed:  758

✓ Loaded 8080 samples from train_plus_val_norm.jsonl

TOKEN LIMIT VIOLATIONS

Articles > 768 tokens: 0

Summaries > 256 tokens: 0

Samples > 1024 tokens: 0

(Note: showing first 10 per category)

SAMPLE 1
Text tokens:     755
Summary tokens:  41
Combined tokens: 806
No truncation needed

SAMPLE 2
Text tokens:     361
Summary tokens:  41
Combined tokens: 412
No truncation needed

SAMPLE 3
Text tokens:     750
Summary tokens:  40
Combined tokens: 800
No truncation needed

